In [2]:
import os

print(os.getcwd())

C:\Users\manda\Healthcare_analytics_project\notebooks


In [3]:
import pandas as pd
df = pd.read_excel(r"C:\Users\manda\Healthcare_analytics_project\data\hospital_final_dataset.xlsx")
categories = sorted(df["APR MDC Description"].dropna().unique())

print(categories)
season_mapping = {
    "Diseases and Disorders of the Respiratory System": {
        "Profile": "WIN",
        "Pattern": "Winter Peak",
        "Justification": "Derived operational assumption based on seasonal respiratory illness patterns."
    },

    "Diseases and Disorders of the Digestive System": {
        "Profile": "SUM",
        "Pattern": "Summer Peak",
        "Justification": "Derived operational assumption for workload modeling."
    },

    "Infectious and Parasitic Diseases, Systemic or Unspecified Sites": {
        "Profile": "MON",
        "Pattern": "Monsoon Peak",
        "Justification": "Derived operational assumption based on seasonal infectious disease patterns."
    }
}

['ALCOHOL/DRUG USE AND ALCOHOL/DRUG INDUCED ORGANIC MENTAL DISORDERS', 'BURNS', 'DISEASES AND DISORDERS OF THE BLOOD AND BLOOD FORMING ORGANS AND IMMUNOLOGICAL DISORDERS', 'DISEASES AND DISORDERS OF THE CIRCULATORY SYSTEM', 'DISEASES AND DISORDERS OF THE DIGESTIVE SYSTEM', 'DISEASES AND DISORDERS OF THE EAR, NOSE, MOUTH AND THROAT', 'DISEASES AND DISORDERS OF THE EYE', 'DISEASES AND DISORDERS OF THE FEMALE REPRODUCTIVE SYSTEM', 'DISEASES AND DISORDERS OF THE HEPATOBILIARY SYSTEM AND PANCREAS', 'DISEASES AND DISORDERS OF THE KIDNEY AND URINARY TRACT', 'DISEASES AND DISORDERS OF THE MALE REPRODUCTIVE SYSTEM', 'DISEASES AND DISORDERS OF THE MUSCULOSKELETAL SYSTEM AND CONNECTIVE TISSUE', 'DISEASES AND DISORDERS OF THE NERVOUS SYSTEM', 'DISEASES AND DISORDERS OF THE RESPIRATORY SYSTEM', 'DISEASES AND DISORDERS OF THE SKIN, SUBCUTANEOUS TISSUE AND BREAST', 'ENDOCRINE, NUTRITIONAL AND METABOLIC DISEASES AND DISORDERS', 'FACTORS INFLUENCING HEALTH STATUS AND OTHER CONTACTS WITH HEALTH SERVICES

In [8]:
season_mapping = {
    "DISEASES AND DISORDERS OF THE RESPIRATORY SYSTEM": {
        "Season Profile": "Winter",
        "Seasonal Pattern": "Winter Peak",
        "Justification": "Derived operational assumption based on increased respiratory workload during colder months."
    },

    "INFECTIOUS AND PARASITIC DISEASES (SYSTEMIC OR UNSPECIFIED SITES)": {
        "Season Profile": "Monsoon",
        "Seasonal Pattern": "Monsoon Peak",
        "Justification": "Derived operational assumption based on increased infectious disease workload during the rainy season."
    },

    "DISEASES AND DISORDERS OF THE DIGESTIVE SYSTEM": {
        "Season Profile": "Summer",
        "Seasonal Pattern": "Summer Peak",
        "Justification": "Derived operational assumption for digestive disease workload."
    },

    "DISEASES AND DISORDERS OF THE KIDNEY AND URINARY TRACT": {
        "Season Profile": "Summer",
        "Seasonal Pattern": "Summer Peak",
        "Justification": "Derived operational assumption for urinary disease workload."
    }
}

In [9]:
mapping_rows = []

for disease in categories:
    if disease in season_mapping:
        profile = season_mapping[disease]
    else:
        profile = {
    "Season Profile": "Uniform",
    "Seasonal Pattern": "Uniform Distribution",
    "Justification": "No seasonal adjustment applied in the derived operational model."
}

    mapping_rows.append({
        "APR MDC Description": disease,
        "Season Profile": profile["Season Profile"],
        "Seasonal Pattern": profile["Seasonal Pattern"],
        "Justification": profile["Justification"]
    })

mapping_df = pd.DataFrame(mapping_rows)

mapping_df.head()

,APR MDC Description,Season Profile,Seasonal Pattern,Justification
0,ALCOHOL/DRUG USE AND ALCOHOL/DRUG INDUCED ORGA...,Uniform,Uniform Distribution,No seasonal adjustment applied in the derived ...
1,BURNS,Uniform,Uniform Distribution,No seasonal adjustment applied in the derived ...
2,DISEASES AND DISORDERS OF THE BLOOD AND BLOOD ...,Uniform,Uniform Distribution,No seasonal adjustment applied in the derived ...
3,DISEASES AND DISORDERS OF THE CIRCULATORY SYSTEM,Uniform,Uniform Distribution,No seasonal adjustment applied in the derived ...
4,DISEASES AND DISORDERS OF THE DIGESTIVE SYSTEM,Summer,Summer Peak,Derived operational assumption for digestive d...


In [10]:
mapping_df

,APR MDC Description,Season Profile,Seasonal Pattern,Justification
0,ALCOHOL/DRUG USE AND ALCOHOL/DRUG INDUCED ORGA...,Uniform,Uniform Distribution,No seasonal adjustment applied in the derived ...
1,BURNS,Uniform,Uniform Distribution,No seasonal adjustment applied in the derived ...
2,DISEASES AND DISORDERS OF THE BLOOD AND BLOOD ...,Uniform,Uniform Distribution,No seasonal adjustment applied in the derived ...
3,DISEASES AND DISORDERS OF THE CIRCULATORY SYSTEM,Uniform,Uniform Distribution,No seasonal adjustment applied in the derived ...
4,DISEASES AND DISORDERS OF THE DIGESTIVE SYSTEM,Summer,Summer Peak,Derived operational assumption for digestive d...
5,"DISEASES AND DISORDERS OF THE EAR, NOSE, MOUTH...",Uniform,Uniform Distribution,No seasonal adjustment applied in the derived ...
6,DISEASES AND DISORDERS OF THE EYE,Uniform,Uniform Distribution,No seasonal adjustment applied in the derived ...
7,DISEASES AND DISORDERS OF THE FEMALE REPRODUCT...,Uniform,Uniform Distribution,No seasonal adjustment applied in the derived ...
8,DISEASES AND DISORDERS OF THE HEPATOBILIARY SY...,Uniform,Uniform Distribution,No seasonal adjustment applied in the derived ...
9,DISEASES AND DISORDERS OF THE KIDNEY AND URINA...,Summer,Summer Peak,Derived operational assumption for urinary dis...


In [11]:
seasonal_weights = pd.DataFrame({
    "Profile": ["Uniform", "Winter", "Summer", "Monsoon"],
    "Jan": [8.33, 14, 6, 5],
    "Feb": [8.33, 13, 6, 5],
    "Mar": [8.33, 10, 7, 6],
    "Apr": [8.33, 8, 9, 7],
    "May": [8.33, 6, 11, 8],
    "Jun": [8.33, 5, 12, 11],
    "Jul": [8.33, 5, 11, 14],
    "Aug": [8.33, 5, 10, 14],
    "Sep": [8.33, 6, 9, 12],
    "Oct": [8.33, 8, 7, 9],
    "Nov": [8.33, 10, 6, 5],
    "Dec": [8.37, 10, 6, 4]
})

seasonal_weights

,Profile,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec
0,Uniform,8.33,8.33,8.33,8.33,8.33,8.33,8.33,8.33,8.33,8.33,8.33,8.37
1,Winter,14.00,13.00,10.00,8.00,6.00,5.00,5.00,5.00,6.00,8.00,10.00,10.00
2,Summer,6.00,6.00,7.00,9.00,11.00,12.00,11.00,10.00,9.00,7.00,6.00,6.00
3,Monsoon,5.00,5.00,6.00,7.00,8.00,11.00,14.00,14.00,12.00,9.00,5.00,4.00


In [12]:
methodology = pd.DataFrame({
    "Item": [
        "Dataset",
        "Original Time Information",
        "Missing Information",
        "Solution",
        "Disease Classification",
        "Operational Factors",
        "Purpose",
        "Original Dataset Modified?"
    ],
    "Description": [
        "hospital_final_dataset.xlsx",
        "Only Discharge Year available",
        "Admission Month and Discharge Month are not available",
        "Derived Monthly Operational Model",
        "APR MDC Description",
        "Length of Stay, Severity Score, Mortality Risk Score, Emergency Flag, Medical/Surgical",
        "Generate Monthly Operational Trends for Tableau Dashboard",
        "No"
    ]
})

methodology

,Item,Description
0,Dataset,hospital_final_dataset.xlsx
1,Original Time Information,Only Discharge Year available
2,Missing Information,Admission Month and Discharge Month are not av...
3,Solution,Derived Monthly Operational Model
4,Disease Classification,APR MDC Description
5,Operational Factors,"Length of Stay, Severity Score, Mortality Risk..."
6,Purpose,Generate Monthly Operational Trends for Tablea...
7,Original Dataset Modified?,No


In [13]:
output_file = "../data/disease_season_mapping.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    mapping_df.to_excel(writer, sheet_name="Disease_Profile", index=False)
    seasonal_weights.to_excel(writer, sheet_name="Seasonal_Weights", index=False)
    methodology.to_excel(writer, sheet_name="Methodology", index=False)

print(f"✅ Workbook created successfully: {output_file}")

✅ Workbook created successfully: ../data/disease_season_mapping.xlsx
